In [ ]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo11n.pt")

# Train a model
model.train(data="dataset/bosch-4k-15c/data.yaml",
            epochs=10, 
            batch=16, 
            imgsz=640)

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("runs/detect/train3/weights/best.pt")

# Test a model on a video and save the result as a video

cap = cv2.VideoCapture("videos/21-09-17-13-00-00.mp4")

# Get video properties
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# Define video writer for output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # Codec
out = cv2.VideoWriter("output.mp4", fourcc, fps, (frame_width, frame_height))

print(f"Frame width: {frame_width}, Frame height: {frame_height}, FPS: {fps}")

# Process video frame by frame
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO model on the frame
    results = model(frame)

    # Draw bounding boxes and labels on the frame
    for result in results[0].boxes:
        box = result.xyxy[0]  # Bounding box in (x1, y1, x2, y2)
        conf = result.conf[0]  # Confidence score
        cls = int(result.cls[0])  # Class index
        label = f"{model.names[cls]} {conf:.2f}"  # Label with class name and confidence

        # Draw rectangle and label
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # Write the processed frame to the output video
    out.write(frame)

# Release resources
cap.release()
out.release()
cv2.destroyAllWindows()

### Get Object Distance

In [ ]:
import cv2
import math
from ultralytics import YOLO

camera_distance = 5 # (cm)
camera_distance_px = (100, 100) # (px)

object_size_dict = {
    "Stop": 6,
} # (cm)

def get_distance(bounding_box, class_name):
    x1, y1, x2, y2 = bounding_box
    centroid_x = (x1 + x2) / 2
    centroid_y = (y1 + y2) / 2
    
    size = object_size_dict.get(class_name, None)
    object_scale = (y2 - y1) / size # px / cm
    
    distance_px = (camera_distance_px[0] - centroid_x) ** 2 + (camera_distance_px[1] - centroid_y) ** 2
    distance = camera_distance * distance_px * object_scale / camera_distance_px[1]
    
    return distance

model = YOLO("runs/detect/train3/weights/best.pt")
